# Kurdistan Nature Dataset Builder 🏔️

Builds a **bigger, cleaner** version of your KND dataset:

```
KND_v2/
├── Dukan Lake/
│   ├── dukan_lake_001.jpg
│   ├── dukan_lake_002.jpg
│   └── dukan_info.txt        (Location / Type / Description — same format as yours)
├── Shanidar Cave/
│   └── ...
└── places.xlsx               (master sheet, same columns as places_template.xlsx)
```

**Pipeline:** search & download images from the web → validate → remove exact/near duplicates → **GPU CLIP filter** (keeps real landscape photos of the place, drops maps, logos, text screenshots, portraits, watermark-heavy stock previews) → write `info.txt` files + Excel → zip.

**Requirements:** Python 3.9+, NVIDIA GPU with CUDA (falls back to CPU automatically, just slower), internet connection.

> ⚠️ Images downloaded from the web belong to their owners. This dataset is fine for **personal study / academic research / model training experiments**, but don't redistribute the images commercially.

## 1. Install dependencies (run once)

In [ ]:
%pip install -q ddgs requests pillow imagehash pandas openpyxl tqdm matplotlib
# PyTorch: if you don't already have a CUDA build, install one matching your driver, e.g.:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
%pip install -q torch torchvision transformers

In [ ]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'   # silence Windows symlink warning

import io, re, time, json, shutil, hashlib, random, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True
Image.MAX_IMAGE_PIXELS = 200_000_000

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Configure the places

All 24 places from your original dataset **plus ~16 new ones**. Each entry has the same metadata fields as your `info.txt` / `places_template.xlsx`, plus several search queries for image diversity. Add or remove places freely — the rest of the notebook adapts automatically.

In [ ]:
# Per-place target AFTER cleaning/filtering. Raise for a bigger dataset.
TARGET_IMAGES_PER_PLACE = 25
# How many raw candidates to fetch per place (fetch extra because filtering removes many)
RAW_CANDIDATES_PER_PLACE = 90

MIN_WIDTH, MIN_HEIGHT = 400, 300      # discard tiny thumbnails
OUT_ROOT = Path('KND_v2')
RAW_ROOT = Path('KND_v2_raw')          # staging area before filtering

def P(name, ptype, area, desc, queries):
    return dict(name=name, type=ptype, area=area, desc=desc, queries=queries)

PLACES = [
    # ---------------- original 24 ----------------
    P('Shanidar Cave', 'Cave / Mountain nature', 'Barzan, Erbil',
      'A historic cave surrounded by natural mountain scenery, known for archaeological importance and hiking views.',
      ['Shanidar Cave Kurdistan', 'Shanidar Cave Iraq entrance', 'Shanidar valley Barzan mountains']),
    P('Korek Mountain', 'Mountain / Resort area', 'Khalifan, Soran',
      'A high mountain destination with cool weather, cable cars, snowy winter views, and tourism resorts.',
      ['Korek Mountain Kurdistan', 'Korek Mountain resort snow', 'Korek Mountain teleferic cable car']),
    P('Sami Abdulrahman Park', 'Park / Picnic area', 'Erbil',
      'One of the largest parks in Kurdistan featuring lakes, green spaces, walking paths, and family picnic areas.',
      ['Sami Abdulrahman Park Erbil', 'Sami Abdulrahman Park lake', 'Sami Abdulrahman Park gardens']),
    P('Rawanduz Canyon', 'Canyon / Nature area', 'Soran',
      'A deep canyon with dramatic cliffs, mountain roads, and scenic viewpoints attracting tourists and photographers.',
      ['Rawanduz Canyon Kurdistan', 'Rawanduz gorge Iraq cliffs', 'Rawanduz canyon viewpoint']),
    P('Barzan Area', 'Mountain nature / Rivers', 'Barzan, Erbil',
      'A peaceful mountain region with rivers, forests, and picnic areas known for natural beauty and cool weather.',
      ['Barzan valley Kurdistan nature', 'Barzan mountains Iraq', 'Great Zab river Barzan']),
    P('Soran Area', 'Mountain nature', 'Soran',
      'A mountainous area surrounded by valleys and green scenery, commonly visited for tourism and outdoor activities.',
      ['Soran Kurdistan mountains nature', 'Soran valley Iraq scenery', 'Soran district landscape']),
    P('Hendrin Mountain', 'Mountain landscape', 'Erbil',
      'A large mountain famous for panoramic natural views, hiking paths, and seasonal greenery.',
      ['Hendrin Mountain Kurdistan', 'Halgurd Sakran Hendrin mountain', 'Hendrin mountain hiking Iraq']),
    P('Dukan Lake', 'Lake / Picnic area', 'Sulaymaniyah',
      'A beautiful lake destination known for boating, fishing, lakeside picnics, and mountain scenery.',
      ['Dukan Lake Kurdistan', 'Dukan Lake Sulaymaniyah boats', 'Dukan lake sunset mountains']),
    P('Azmar Mountain', 'Mountain / Hiking area', 'Sulaymaniyah',
      'A mountain overlooking Sulaymaniyah city, popular for hiking, exercise, and scenic city views.',
      ['Azmar Mountain Sulaymaniyah', 'Azmar Goizha mountain view', 'Azmar mountain hiking']),
    P('Ahmad Awa Waterfall', 'Waterfall / Forest area', 'Hawraman, Sulaymaniyah',
      'A well-known waterfall surrounded by forests, restaurants, and picnic areas with cool mountain air.',
      ['Ahmad Awa waterfall Kurdistan', 'Ahmadawa waterfall Iraq', 'Ahmad Awa Hawraman waterfall']),
    P('Hawraman Region', 'Mountain villages / Nature', 'Halabja',
      'A mountainous cultural region famous for stone terraced villages, rivers, green landscapes, and traditional architecture.',
      ['Hawraman Kurdistan terraced village', 'Hawraman Takht mountains', 'Hawraman valley Iraq nature']),
    P('Qaradagh Area', 'Hills / Nature area', 'Sulaymaniyah',
      'A hilly natural region with open landscapes, valleys, and peaceful outdoor environments.',
      ['Qaradagh Sulaymaniyah nature', 'Qaradagh mountains Kurdistan', 'Qaradagh hills landscape']),
    P('Sirwan River Area', 'River / Nature', 'Sulaymaniyah / Halabja',
      'River areas with flowing water, surrounding greenery, and popular picnic and fishing locations.',
      ['Sirwan river Kurdistan', 'Sirwan river Halabja nature', 'Diyala Sirwan river valley']),
    P('Amedi', 'Mountain town / Nature', 'Duhok',
      'A historic mountain town on a plateau surrounded by cliffs and green scenery, famous for natural beauty and cultural tourism.',
      ['Amedi town Kurdistan', 'Amadiya Iraq aerial plateau', 'Amedi Duhok mountains']),
    P('Lalish', 'Valley / Nature area', 'Shekhan, Duhok',
      'A peaceful valley surrounded by mountains and trees, known for natural scenery and spiritual importance.',
      ['Lalish valley Kurdistan', 'Lalish temple valley nature', 'Lalish Shekhan Duhok']),
    P('Akre', 'Mountains / Hiking', 'Duhok',
      'A mountainous region with beautiful hills, hiking areas, and traditional Kurdish landscapes.',
      ['Akre Kurdistan town mountains', 'Akre Newroz mountain', 'Akre Duhok scenery']),
    P('Gali Duhok', 'Valley / Nature area', 'Duhok',
      'A green valley area surrounded by mountains, popular for outdoor recreation and family picnics.',
      ['Gali Duhok valley', 'Duhok gorge nature', 'Duhok valley mountains Kurdistan']),
    P('Zakho', 'River / Picnic area', 'Duhok',
      'River areas around Zakho with flowing water, the historic Delal bridge, green nature, and picnic destinations.',
      ['Zakho Delal bridge', 'Zakho Khabur river Kurdistan', 'Zakho nature Iraq']),
    P('Shaqlawa', 'Mountain town / Nature', 'Erbil',
      'A famous mountain town with cool weather, green landscapes, springs, and picnic areas visited during summer.',
      ['Shaqlawa Kurdistan', 'Shaqlawa Safeen mountain', 'Shaqlawa resort nature']),
    P('Darbandikhan Lake', 'Lake / Nature area', 'Darbandikhan, Sulaymaniyah',
      'A large lake surrounded by mountains, popular for boating, fishing, and relaxing outdoor activities.',
      ['Darbandikhan lake Kurdistan', 'Darbandikhan dam Iraq', 'Darbandikhan lake mountains']),
    P('Darband Lake (Ranya)', 'Lake / Nature area', 'Ranya, Raparin',
      'A large lake surrounded by mountains, popular for boating, fishing, relaxing outdoor activities, and spring picnics.',
      ['Darband Ranya lake', 'Ranya lake Kurdistan', 'Darbandi Ranya nature']),
    P('Gali Ali Bag Waterfall', 'Waterfall / Canyon', 'Khalifan, Soran',
      'A large waterfall area surrounded by mountains and valleys, popular for sightseeing, photography, and summer picnics.',
      ['Gali Ali Bag waterfall', 'Geli Ali Beg waterfall Kurdistan', 'Gali Ali Bag canyon Iraq']),
    P('Halgurd Mountain', 'Mountain / Hiking area', 'Choman, Erbil',
      'One of the highest mountains in Iraq, popular for hiking, climbing, snow, and beautiful natural landscapes.',
      ['Halgurd mountain Kurdistan', 'Halgurd peak snow Iraq', 'Mount Halgurd Choman climbing']),
    P('Przhe Mountain', 'Mountain / Hiking area', 'Choman, Erbil',
      'A famous mountain with waterfalls, popular for camping, relaxing, and picnics during all seasons.',
      ['Przhe mountain Choman', 'Przhe waterfall Kurdistan', 'Choman mountains nature Iraq']),
    # ---------------- NEW places ----------------
    P('Bekhal Waterfall', 'Waterfall / Nature area', 'Soran',
      'A famous cascading waterfall on the Hamilton Road, surrounded by cliffs, restaurants, and busy summer picnic spots.',
      ['Bekhal waterfall Kurdistan', 'Bekhal waterfall Soran Iraq', 'Bekhal falls Rawanduz']),
    P('Sulav Resort', 'Waterfall / Mountain resort', 'Amedi, Duhok',
      'A mountain resort near Amedi with springs, small waterfalls, shaded walkways, and traditional restaurants.',
      ['Sulav resort Amedi', 'Sulav waterfall Duhok', 'Sulav Kurdistan nature']),
    P('Zawita Valley', 'Valley / Pine forest', 'Duhok',
      'A green pine-forested valley near Duhok, known for cool air, forest walks, and picnic areas.',
      ['Zawita valley Duhok', 'Zawita pine forest Kurdistan', 'Zawita nature Iraq']),
    P('Gara Mountain', 'Mountain / Snow area', 'Duhok',
      'A high mountain range south of Amedi with snow in winter, forests, and wide panoramic views.',
      ['Gara mountain Duhok', 'Gara mountain snow Kurdistan', 'Matin Gara mountains Iraq']),
    P('Safeen Mountain', 'Mountain landscape', 'Erbil',
      'A long mountain ridge above Shaqlawa, popular for hiking, paragliding, and views over green plains.',
      ['Safeen mountain Kurdistan', 'Safeen mountain Shaqlawa', 'Safin mountain Erbil hiking']),
    P('Piramagrun Mountain', 'Mountain / Hiking area', 'Sulaymaniyah',
      'A striking rocky mountain northwest of Sulaymaniyah, famous for hiking routes, caves, and dramatic peaks.',
      ['Piramagrun mountain', 'Piramagrun Sulaymaniyah hiking', 'Pira Magrun peak Kurdistan']),
    P('Goizha Mountain', 'Mountain / City viewpoint', 'Sulaymaniyah',
      'A mountain overlooking Sulaymaniyah, connected to Azmar, popular for evening walks and panoramic city views.',
      ['Goizha mountain Sulaymaniyah', 'Goizha viewpoint city', 'Goizha Azmar Kurdistan']),
    P('Penjwen Area', 'Mountain nature / Border highlands', 'Sulaymaniyah',
      'A highland border region with green hills, orchards, snow in winter, and cool summer weather.',
      ['Penjwen Kurdistan nature', 'Penjwen mountains Sulaymaniyah', 'Penjwen snow hills']),
    P('Mergapan Valley', 'Valley / Spring nature', 'Sulaymaniyah',
      'A green valley famous for spring wildflowers, streams, and camping between the mountains.',
      ['Mergapan valley Kurdistan', 'Mergapan Sulaymaniyah spring', 'Mergapan nature Iraq']),
    P('Dore Canyon', 'Canyon / Nature area', 'Barzan, Erbil',
      'A narrow dramatic canyon in the Barzan area with steep rock walls and a river running through.',
      ['Dore canyon Barzan', 'Dore canyon Kurdistan', 'Dora canyon Erbil Iraq']),
    P('Great Zab River', 'River / Nature', 'Erbil / Duhok',
      'A major river flowing through the mountains of Kurdistan, with rafting spots, green banks, and fishing areas.',
      ['Great Zab river Kurdistan', 'Zab river Iraq mountains', 'Greater Zab rafting']),
    P('Khalifan Area', 'Mountain nature', 'Soran',
      'A mountainous district on the road to Soran with valleys, waterfalls, and green scenery.',
      ['Khalifan Kurdistan nature', 'Khalifan valley Soran', 'Khalifan mountains Iraq']),
    P('Chami Rezan Valley', 'Valley / Archaeological nature', 'Sulaymaniyah',
      'A quiet valley with caves and archaeological sites, surrounded by hills and seasonal greenery.',
      ['Chami Rezan valley', 'Chami Razan Sulaymaniyah', 'Chami Rezan caves Kurdistan']),
    P('Sartka Mountain', 'Mountain / Camping area', 'Sulaymaniyah',
      'A popular camping and paragliding mountain area near Dukan with wide views over the lake region.',
      ['Sartka mountain Kurdistan', 'Sartka Dukan camping', 'Sartka paragliding Sulaymaniyah']),
    P('Barzinja Area', 'Hills / Springs', 'Sulaymaniyah',
      'A green area of hills and natural springs east of Sulaymaniyah, popular for weekend picnics.',
      ['Barzinja Kurdistan nature', 'Barzinja springs Sulaymaniyah', 'Barzinja hills Iraq']),
    P('Smaquli Valley', 'Valley / Nature area', 'Erbil',
      'A green valley east of Erbil surrounded by mountains, with streams and spring wildflowers.',
      ['Smaquli valley Erbil', 'Smaquli Kurdistan nature', 'Smaquli mountains Iraq']),
]

print(f'{len(PLACES)} places configured')
print(f'Target dataset size ≈ {len(PLACES) * TARGET_IMAGES_PER_PLACE} images')

## 3. Search & download raw images

Uses DuckDuckGo image search (`ddgs`, no API key needed). Downloads are parallel, with retries and a browser User-Agent. Re-running the cell skips places that already have enough raw images.

In [ ]:
try:
    from ddgs import DDGS               # new package name (2025+)
except ImportError:
    from duckduckgo_search import DDGS  # older package name

HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0 Safari/537.36'}

def slugify(s):
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')

def search_image_urls(place, max_urls):
    """Collect candidate image URLs from all queries for a place."""
    urls, seen = [], set()
    per_query = max(15, max_urls // len(place['queries']) + 5)
    for q in place['queries']:
        for attempt in range(3):
            try:
                with DDGS() as ddgs:
                    results = ddgs.images(q, max_results=per_query, size='Large')
                    for r in results or []:
                        u = r.get('image')
                        if u and u not in seen:
                            seen.add(u); urls.append(u)
                break
            except Exception as e:
                wait = 3 * (attempt + 1)
                print(f'  search retry ({q!r}): {e} — waiting {wait}s')
                time.sleep(wait)
        time.sleep(1.0)  # be polite, avoid rate limits
    random.shuffle(urls)
    return urls[:max_urls]

def download_one(url, dest_dir, idx):
    try:
        r = requests.get(url, headers=HEADERS, timeout=15, stream=True)
        r.raise_for_status()
        data = r.content
        if len(data) < 8_000:          # skip tiny files / tracking pixels
            return None
        img = Image.open(io.BytesIO(data))
        img.load()
        if img.width < MIN_WIDTH or img.height < MIN_HEIGHT:
            return None
        img = img.convert('RGB')       # normalizes webp/avif/png/CMYK → RGB jpg
        # cap very large images to keep the dataset light
        if max(img.size) > 2200:
            img.thumbnail((2200, 2200), Image.LANCZOS)
        out = dest_dir / f'raw_{idx:04d}.jpg'
        img.save(out, 'JPEG', quality=90)
        return out
    except Exception:
        return None

RAW_ROOT.mkdir(exist_ok=True)

for place in tqdm(PLACES, desc='Places'):
    dest = RAW_ROOT / place['name']
    dest.mkdir(exist_ok=True)
    existing = len(list(dest.glob('raw_*.jpg')))
    if existing >= RAW_CANDIDATES_PER_PLACE * 0.6:
        print(f"✓ {place['name']}: {existing} raw images already, skipping")
        continue
    urls = search_image_urls(place, RAW_CANDIDATES_PER_PLACE)
    print(f"{place['name']}: {len(urls)} candidate URLs")
    ok = 0
    with ThreadPoolExecutor(max_workers=12) as ex:
        futures = {ex.submit(download_one, u, dest, i): u for i, u in enumerate(urls)}
        for f in as_completed(futures):
            if f.result() is not None:
                ok += 1
    print(f"  downloaded {ok} valid images")

total_raw = sum(len(list((RAW_ROOT / p['name']).glob('raw_*.jpg'))) for p in PLACES)
print(f'\nTotal raw images: {total_raw}')

## 4. Remove exact & near-duplicate images (perceptual hashing)

Web search returns the same photo many times at different sizes. MD5 catches exact copies; perceptual hash (`phash`) catches resized/re-compressed copies.

In [ ]:
import imagehash

PHASH_DISTANCE = 6   # <= this Hamming distance counts as duplicate

def dedup_folder(folder):
    files = sorted(folder.glob('raw_*.jpg'))
    md5s, hashes, removed = set(), [], 0
    for f in files:
        try:
            data = f.read_bytes()
            m = hashlib.md5(data).hexdigest()
            if m in md5s:
                f.unlink(); removed += 1; continue
            h = imagehash.phash(Image.open(io.BytesIO(data)))
            if any(h - h2 <= PHASH_DISTANCE for h2 in hashes):
                f.unlink(); removed += 1; continue
            md5s.add(m); hashes.append(h)
        except Exception:
            f.unlink(missing_ok=True); removed += 1
    return removed, len(list(folder.glob('raw_*.jpg')))

for place in tqdm(PLACES, desc='Deduplicating'):
    removed, kept = dedup_folder(RAW_ROOT / place['name'])
    print(f"{place['name']}: removed {removed} dupes, {kept} remain")

## 5. GPU CLIP filtering — keep only real photos of the place

This is where your GPU earns its keep. OpenAI CLIP scores each image against:
- a **positive prompt** describing the place (e.g. *"a photo of a waterfall in mountains"*), and
- **negative prompts**: map, logo, text screenshot, food dish, portrait/selfie, indoor room, cartoon.

Images whose best match is a negative prompt (or whose positive score is weak) are dropped. The survivors are ranked by positive score and the top `TARGET_IMAGES_PER_PLACE` are kept.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

# use_safetensors=True avoids the torch<2.6 CVE-2025-32434 loading restriction
clip_model = CLIPModel.from_pretrained(
    'openai/clip-vit-base-patch32',
    use_safetensors=True,
).to(DEVICE).eval()
clip_proc = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

NEGATIVE_PROMPTS = [
    'a map or satellite view',
    'a logo, banner or graphic design',
    'a screenshot of text or a document',
    'a close-up portrait or selfie of a person',
    'a plate of food in a restaurant',
    'the interior of a room or shop',
    'a cartoon, drawing or illustration',
]

TYPE_HINTS = {  # keyword in place type -> scene phrase for the positive prompt
    'waterfall': 'a waterfall in the mountains',
    'lake': 'a lake surrounded by mountains',
    'river': 'a river flowing through nature',
    'canyon': 'a rocky canyon with cliffs',
    'cave': 'a cave entrance in a mountainside',
    'valley': 'a green valley between mountains',
    'park': 'a large green park with trees and paths',
    'town': 'a small town in the mountains',
    'village': 'traditional villages on a mountainside',
    'hill': 'green hills and open landscape',
    'mountain': 'a mountain landscape',
}

def positive_prompt(place):
    t = place['type'].lower()
    for kw, phrase in TYPE_HINTS.items():
        if kw in t:
            return f'an outdoor photo of {phrase} in Kurdistan'
    return 'an outdoor photo of natural scenery in Kurdistan'

@torch.no_grad()
def clip_scores(image_paths, prompts, batch_size=32):
    """Return (N_images, N_prompts) softmax probabilities."""
    all_probs = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        images = []
        valid_idx = []
        for j, p in enumerate(batch_paths):
            try:
                images.append(Image.open(p).convert('RGB'))
                valid_idx.append(j)
            except Exception:
                pass
        if not images:
            all_probs.extend([None] * len(batch_paths)); continue
        inputs = clip_proc(text=prompts, images=images, return_tensors='pt', padding=True).to(DEVICE)
        probs = clip_model(**inputs).logits_per_image.softmax(dim=1).cpu()
        out = [None] * len(batch_paths)
        for k, j in enumerate(valid_idx):
            out[j] = probs[k]
        all_probs.extend(out)
    return all_probs

OUT_ROOT.mkdir(exist_ok=True)
report = []

for place in tqdm(PLACES, desc='CLIP filtering'):
    src = RAW_ROOT / place['name']
    files = sorted(src.glob('raw_*.jpg'))
    if not files:
        report.append((place['name'], 0, 0)); continue
    pos = positive_prompt(place)
    prompts = [pos] + NEGATIVE_PROMPTS
    probs = clip_scores(files, prompts)

    scored = []
    for f, p in zip(files, probs):
        if p is None:
            continue
        pos_score = p[0].item()
        best = int(p.argmax())
        # keep if the positive prompt wins, or at least gets a solid share
        if best == 0 or pos_score >= 0.35:
            scored.append((pos_score, f))
    scored.sort(reverse=True, key=lambda x: x[0])
    keep = scored[:TARGET_IMAGES_PER_PLACE]

    dest = OUT_ROOT / place['name']
    dest.mkdir(exist_ok=True)
    slug = slugify(place['name'])
    for i, (score, f) in enumerate(keep, 1):
        shutil.copy2(f, dest / f'{slug}_{i:03d}.jpg')
    report.append((place['name'], len(files), len(keep)))

print('\n{:<28} {:>6} {:>6}'.format('Place', 'raw', 'kept'))
for name, raw, kept in report:
    flag = '  ⚠ low!' if kept < 10 else ''
    print(f'{name:<28} {raw:>6} {kept:>6}{flag}')
print(f'\nFinal dataset images: {sum(k for _,_,k in report)}')

## 6. Write `info.txt` files + master `places.xlsx`

Same format as your original dataset: each folder gets an `<slug>_info.txt` with Location / Type / Description, and the root gets an Excel sheet with the same columns as `places_template.xlsx` (plus an image count).

In [ ]:
rows = []
for place in PLACES:
    dest = OUT_ROOT / place['name']
    if not dest.exists():
        continue
    n_imgs = len(list(dest.glob('*.jpg')))
    info = (
        f"Location: {place['area']}\n\n\n"
        f"Type: {place['type']}\n\n\n"
        f"Description: {place['desc']}\n"
    )
    (dest / f"{slugify(place['name'])}_info.txt").write_text(info, encoding='utf-8')
    rows.append({
        'Place Name': place['name'],
        'Type of Place': place['type'],
        'Description': place['desc'],
        'Area': place['area'],
        'Image Count': n_imgs,
    })

df = pd.DataFrame(rows)
df.to_excel(OUT_ROOT / 'places.xlsx', sheet_name='Places', index=False)
df

## 7. Preview a random sample

In [ ]:
import matplotlib.pyplot as plt

sample_places = random.sample([p for p in PLACES if (OUT_ROOT / p['name']).exists()], k=min(6, len(PLACES)))
fig, axes = plt.subplots(len(sample_places), 4, figsize=(14, 3.2 * len(sample_places)))
for r, place in enumerate(sample_places):
    imgs = sorted((OUT_ROOT / place['name']).glob('*.jpg'))[:4]
    for c in range(4):
        ax = axes[r, c] if len(sample_places) > 1 else axes[c]
        ax.axis('off')
        if c < len(imgs):
            ax.imshow(Image.open(imgs[c]))
            if c == 0:
                ax.set_title(place['name'], loc='left', fontsize=11)
plt.tight_layout(); plt.show()

## 8. Zip the final dataset

In [ ]:
zip_path = 'KND_v2.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUT_ROOT.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(OUT_ROOT.parent))
size_mb = os.path.getsize(zip_path) / 1e6
print(f'Done → {zip_path} ({size_mb:.1f} MB)')
print(f"Places: {len(rows)} | Images: {int(df['Image Count'].sum())}")
# Optional: free disk space by deleting the raw staging folder
# shutil.rmtree(RAW_ROOT)

## Tips for going even bigger

- **More images per place:** raise `TARGET_IMAGES_PER_PLACE` (e.g. 50) and `RAW_CANDIDATES_PER_PLACE` (e.g. 200), then re-run cells 3→8. Existing raw images are kept, so it only downloads the extra ones.
- **More places:** append new `P(...)` entries in cell 2 — Kurdish + English query variants (e.g. "گەلی عەلی بەگ" alongside "Gali Ali Bag") often surface different photos.
- **Stricter quality:** lower the `0.35` positive-score threshold in cell 5 to keep more, or raise it (e.g. 0.5) plus use the larger model `openai/clip-vit-large-patch14` for cleaner results (needs ~2 GB VRAM more).
- **Cross-place duplicate check:** some famous photos appear under several place names; you can run the phash dedup once over the whole `KND_v2` tree at the end.
- **Rate limits:** if DuckDuckGo starts returning empty results, wait a few minutes or increase the `time.sleep(1.0)` between queries.